In [1]:
import pandas as pd
import json
import openpyxl

In [2]:
# load data
orbis_match_1 = pd.read_excel('../data/raw/orbis/match_treated/Export_orbis_firm_match_1.xlsx')
orbis_match_2 = pd.read_excel('../data/raw/orbis/match_treated/Export_orbis_firm_match_2.xlsx')
orbis_match_3 = pd.read_excel('../data/raw/orbis/match_treated/Export_orbis_firm_match_3.xlsx', sheet_name='Page 1')
orbis_match_4 = pd.read_excel('../data/raw/orbis/match_treated/Export_orbis_firm_match_3.xlsx', sheet_name='page_2')

In [3]:
# create master orbis match df
orbis_match_combined = pd.concat([orbis_match_1, orbis_match_2, orbis_match_3, orbis_match_4], ignore_index=True)
orbis_match_combined = orbis_match_combined.dropna(subset=['Ragione sociale corrispondente'])
orbis_match_combined = orbis_match_combined.drop_duplicates(subset=['Denominazione/Ragione sociale', 'Paese'])
orbis_match_combined

,Denominazione/Ragione sociale,Città,Paese,Identifier,Score,BvD ID corrispondente,Ragione sociale corrispondente
0,C.H. Boehringer Sohn AG & Co. KG,Ingelheim am Rhein,Germany,NaN,A,DE6190007083,C.H. BOEHRINGER SOHN AG & CO. KG
1,Boehringer Ingelheim Pharma GmbH & Co. KG,Ingelheim am Rhein,Germany,NaN,A,DE6190143582,BOEHRINGER INGELHEIM PHARMA GMBH & CO. KG
2,Boehringer Ingelheim GmbH,Ingelheim am Rhein,Germany,NaN,A,DE6190007305,BOEHRINGER INGELHEIM GMBH
3,Alkaloids of Australia Pty. Limited,Toowoomba,Australia,NaN,A,AU010675439,ALKALOIDS OF AUSTRALIA PTY. LIMITED
4,Transo-Pharm Holding-AG,Siek,Germany,NaN,A,DE2390275417,TRANSO-PHARM HOLDING-AG
...,...,...,...,...,...,...,...
2051,TRELLEBORG INDUSTRIE SAS,NaN,France,NaN,NaN,NO998234808,TRELLEBORG INDUSTRIE SAS
2052,UTi Nederland B.V.,NaN,Netherlands,NaN,NaN,NL34088111,UTI (NETHERLANDS) HOLDINGS B.V.
2053,Voestalpine Austria Draht GmbH,NaN,Austria,NaN,NaN,AT9030280443,VOESTALPINE WIRE AUSTRIA GMBH
2054,Volvo Car Corporation,NaN,Sweden,NaN,NaN,BR05721601000107,VOLVO CAR CORPORATION


In [4]:
# import treated firms list
with open('../data/processed/treat_firm_list.json', 'r') as f:
    treated_firms = json.load(f)

# filter orbis match df with treated firms list
orbis_match_treated = orbis_match_combined[orbis_match_combined['Denominazione/Ragione sociale'].isin(treated_firms)]

# rename cols
orbis_match_treated = orbis_match_treated.rename(columns={
    'Denominazione/Ragione sociale' : 'firm_name',
    'BvD ID corrispondente' : 'bvd_id'
})
orbis_match_treated

,firm_name,Città,Paese,Identifier,Score,bvd_id,Ragione sociale corrispondente
0,C.H. Boehringer Sohn AG & Co. KG,Ingelheim am Rhein,Germany,NaN,A,DE6190007083,C.H. BOEHRINGER SOHN AG & CO. KG
1,Boehringer Ingelheim Pharma GmbH & Co. KG,Ingelheim am Rhein,Germany,NaN,A,DE6190143582,BOEHRINGER INGELHEIM PHARMA GMBH & CO. KG
2,Boehringer Ingelheim GmbH,Ingelheim am Rhein,Germany,NaN,A,DE6190007305,BOEHRINGER INGELHEIM GMBH
3,Alkaloids of Australia Pty. Limited,Toowoomba,Australia,NaN,A,AU010675439,ALKALOIDS OF AUSTRALIA PTY. LIMITED
4,Transo-Pharm Holding-AG,Siek,Germany,NaN,A,DE2390275417,TRANSO-PHARM HOLDING-AG
...,...,...,...,...,...,...,...
2046,Hynix Semiconductor United Kingdom Ltd,NaN,United Kingdom,NaN,NaN,GB03077981,SK HYNIX UK LIMITED
2047,Société Nationale Chemins de fer Belgique SA -...,NaN,Belgium,NaN,NaN,BE2281516016,SNCB - NMBS
2050,Toll Global Forwarding Limited,NaN,Hong Kong,NaN,NaN,HK0000023908,TOLL GLOBAL FORWARDING LIMITED
2052,UTi Nederland B.V.,NaN,Netherlands,NaN,NaN,NL34088111,UTI (NETHERLANDS) HOLDINGS B.V.


In [5]:
orbis_match_treated.to_excel('../data/interim/orbis_match_treated.xlsx', index=False)

In [6]:
# treated firms missing from orbis match df
missing_firms = [f for f in treated_firms if f not in orbis_match_treated['firm_name'].values]
len(missing_firms)

17

In [7]:
duplicates = orbis_match_treated[orbis_match_treated.duplicated(subset=['bvd_id'], keep=False)]
duplicates.sort_values(by='firm_name')

,firm_name,Città,Paese,Identifier,Score,bvd_id,Ragione sociale corrispondente
140,Bonduelle SA,Renescure,France,NaN,A,FR447250044,BONDUELLE
536,Bonduelle SAS,Renescure,France,NaN,A,FR447250044,BONDUELLE
137,Bonduelle SCA,Renescure,France,NaN,A,FR447250044,BONDUELLE
829,Chiquita Banana Company BV,Gorinchem,Netherlands,NaN,A,NL20077397,CHIQUITA EUROPE B.V. (Previous name: CHIQUITA ...
1056,Chiquita Brands International Inc,Cincinatti,United States,NaN,A,US356795618L,CHIQUITA BRANDS INTERNATIONAL
828,"Chiquita Brands International, Inc.",Cincinnati,United States,NaN,A,US356795618L,CHIQUITA BRANDS INTERNATIONAL
1057,Chuiquita Banana Company BV,Gorinchem,Netherlands,NaN,A,NL20077397,CHIQUITA EUROPE B.V. (Previous name: CHIQUITA ...
1473,Coats Holdings Ltd,Stockley Park,United Kingdom,NaN,B,GB00104998,COATS HOLDINGS LTD
1186,Coats Holdings Ltd.,Uxbridge,United Kingdom,NaN,A,GB00104998,COATS HOLDINGS LTD
142,Daimler AG,Stuttgart,Germany,NaN,A,DE7330530056,MERCEDES-BENZ GROUP AG (Previous name: DAIMLER...


In [8]:
# get unique bvd_ids for matched firms
treated_bvd_id = orbis_match_treated[['bvd_id']]
treated_bvd_id = treated_bvd_id.drop_duplicates()
treated_bvd_id.to_csv('../data/interim/bvdid_treat.csv', sep="\t", index=False, header=False)

## import orbis data

In [11]:
orbis_treated_df = pd.read_excel('../data/raw/orbis/nace_treat_15.12.25.xlsx', sheet_name='Risultati', dtype=str)
orbis_treated_df

,Unnamed: 0,Ragione socialeCaratteri latini,"Codice NACE Rev. 2, core code (4 cifre)",Stato della quotazione,Numero BvD ID
0,1.,VOLKSWAGEN AG,2910,Quotata,DE2070000543
1,2.,TOYOTA MOTOR CORPORATION,2910,Quotata,JP1180301018771
2,3.,"SAMSUNG ELECTRONICS CO.,LTD.",2630,Quotata,KR1301110006246
3,4.,TOTALENERGIES SE,0610,Quotata,FR542051180
4,5.,FORD MOTOR COMPANY,2910,Quotata,US380549190
...,...,...,...,...,...
724,725.,AGILITY LOGISTICS LTD,NaN,Non quotata,GB*110372533683
725,726.,SANDEN HOLDINGS CORPORATION,NaN,Non quotata,JP*110373259284
726,727.,ALCHEM INTERNATIONAL (H.K.) LIMITED,NaN,Non quotata,HK0001054632
727,728.,CNH INDUSTRIAL N.V.,NaN,Non quotata,GB*110378295516


In [12]:
# rename columns
orbis_treated_df = orbis_treated_df.rename(columns={
    'Ragione socialeCaratteri latini' : 'firm_name',
    'Codice NACE Rev. 2, core code (4 cifre)' : 'nace_core',
    'Stato della quotazione' : 'quoted',
    'Numero BvD ID' : 'bvd_id'
})

orbis_treated_df

,Unnamed: 0,firm_name,nace_core,quoted,bvd_id
0,1.,VOLKSWAGEN AG,2910,Quotata,DE2070000543
1,2.,TOYOTA MOTOR CORPORATION,2910,Quotata,JP1180301018771
2,3.,"SAMSUNG ELECTRONICS CO.,LTD.",2630,Quotata,KR1301110006246
3,4.,TOTALENERGIES SE,0610,Quotata,FR542051180
4,5.,FORD MOTOR COMPANY,2910,Quotata,US380549190
...,...,...,...,...,...
724,725.,AGILITY LOGISTICS LTD,NaN,Non quotata,GB*110372533683
725,726.,SANDEN HOLDINGS CORPORATION,NaN,Non quotata,JP*110373259284
726,727.,ALCHEM INTERNATIONAL (H.K.) LIMITED,NaN,Non quotata,HK0001054632
727,728.,CNH INDUSTRIAL N.V.,NaN,Non quotata,GB*110378295516


In [13]:
# extract nace codes of treated firms
treat_listed = orbis_treated_df[orbis_treated_df['quoted'] == 'Quotata']
nace_listed = treat_listed[['nace_core']].drop_duplicates()
nace_listed.to_csv('../data/interim/nace_treat.csv', sep="\t", index=False)

In [ ]:
cartel_firms = pd.read_excel('../data/naics.xlsx', sheet_name='Risultati', dtype=str)
# Extract unique NAICS codes
unique_codes = cartel_firms['NAICS 2022, codici/e primari/o'].dropna().unique()

# Convert to DataFrame
unique_df = pd.DataFrame(unique_codes, columns=['NAICS_code'])

# Save to Excel
output_path = '../data/unique_naics_codes.xlsx'
unique_df.to_excel(output_path, index=False)